# Using geopoints data

In [1]:
import earthkit.data as ekd

ds = ekd.from_source("file", "metadata.gpt")
ds

path,metadata.gpt
size,1.6 KiB
types,"pandas, xarray, featurelist"


The data can be converted into a Pandas DataFrame with `to_pandas()`. The date and time columns are merged into a single `date` datetime column, and the geopoints missing value (3e38) is converted to NaN.

In [2]:
df = ds.to_pandas()
df.head()

,latitude,longitude,level,date,value
0,55.01,8.41,0.2,2013-08-04 12:00:00,294.4
1,54.33,8.60,0.2,2013-08-04 12:00:00,296.9
2,53.71,7.15,0.2,2013-08-04 12:00:00,297.1
3,53.53,8.58,0.2,2013-08-04 12:00:00,295.8
4,53.87,8.71,0.2,2013-08-04 12:00:00,294.8


## NCOLS format

In [3]:
ds = ekd.from_source("file", "ncols_stnid.gpt")
ds.to_pandas()

,stnid,latitude,longitude,level,date,_v1,_v2,_v3
0,heathrow,32.55,35.85,10.0,2012-02-18 06:00:00,273.9,29.0,7.0
1,oxford,31.72,35.98,20.0,2012-02-18 18:00:00,274.9,28.0,1.0
2,texas,51.93,8.32,30.0,2014-02-18 12:00:00,278.9,27.0,6.0
3,aberdeen,41.10,20.82,11.0,2015-02-18 12:00:00,279.9,26.0,0.5


## Geopointset

In [4]:
s = ekd.from_source("file", "geopointset.gpts")
s

path,geopointset.gpts
size,781
types,"pandas, xarray, featurelist"


`to_pandas()` stacks the data of all the blocks into a single DataFrame. Columns that are missing in a block are filled with NaN:

In [5]:
s.to_pandas()

,latitude,longitude,level,date,value
0,69.6523,18.9057,0.0,2013-05-12 00:00:00,100869.86250
1,63.4882,10.8795,0.0,2013-05-12 00:00:00,100282.33920
2,63.5657,10.6940,0.0,2013-05-12 00:00:00,100241.16660
3,61.2928,5.0443,0.0,2013-05-12 00:00:00,99852.18932
4,60.8200,23.5000,0.0,2013-05-12 06:00:00,101045.80000
5,55.0100,8.4100,0.0,2013-05-13 00:00:00,100949.18090
6,54.3300,8.6200,0.0,2013-05-13 00:00:00,101027.91010
7,53.7100,7.1500,0.0,2013-05-13 00:00:00,100846.61900


## Conversion to Xarray

In [6]:
ds = ekd.from_source("file", "metadata.gpt")
print(ds.to_xarray())

<xarray.Dataset> Size: 2kB
Dimensions:    (index: 44)
Coordinates:
  * index      (index) int64 352B 0 1 2 3 4 5 6 7 8 ... 36 37 38 39 40 41 42 43
Data variables:
    latitude   (index) float64 352B 55.01 54.33 53.71 ... 48.45 48.11 47.88
    longitude  (index) float64 352B 8.41 8.6 7.15 8.58 ... 7.83 8.41 8.76 8.0
    level      (index) float64 352B 0.2 0.2 0.2 0.2 0.2 ... 0.2 0.2 0.2 0.2 0.2
    date       (index) datetime64[ns] 352B 2013-08-04T12:00:00 ... 2013-08-04...
    value      (index) float64 352B 294.4 296.9 297.1 ... 294.5 294.8 290.5


## Featurelist

In [7]:
fl = s.to_featurelist()
len(fl)

3

A feature gives access to the format and the columns of its block and to its number of points. Only the second block of our geopointset has metadata:

In [8]:
f = fl[1]
print(f.format, f.columns, len(f))
print(f.metadata())
print(fl[0].metadata())

TRADITIONAL ['latitude', 'longitude', 'level', 'date', 'time', 'value'] 1
{'param': 'geopotential'}
{}


`ls()` summarises the metadata of the features (a feature without a key shows None), and `sel()` selects the features by metadata:

In [9]:
fl.ls()

,param
0,None
1,geopotential
2,None


In [10]:
sub = fl.sel(param="geopotential")
len(sub)

1

The data of a feature is available with `to_pandas()`:

In [11]:
sub[0].to_pandas()

,latitude,longitude,level,date,value
0,60.82,23.5,0.0,2013-05-12 06:00:00,101045.8


### Row mode

With `mode="row"` each feature is a single data point (a row of the DataFrame). The metadata of a row is the `#METADATA` of its block plus the values of the row's columns; if a name is in both, the value in the row is used. This is useful to select or sort the points, e.g. by station id:

In [12]:
ds = ekd.from_source("file", "ncols_stnid.gpt")
rl = ds.to_featurelist(mode="row")
rl.sel(stnid="oxford").ls()

,stnid,latitude,longitude,level,date,_v1,_v2,_v3
0,oxford,31.72,35.98,20.0,2012-02-18 18:00:00,274.9,28.0,1.0


In [13]:
rl.order_by(level="descending").ls()

,stnid,latitude,longitude,level,date,_v1,_v2,_v3
0,texas,51.93,8.32,30.0,2014-02-18 12:00:00,278.9,27.0,6.0
1,oxford,31.72,35.98,20.0,2012-02-18 18:00:00,274.9,28.0,1.0
2,aberdeen,41.10,20.82,11.0,2015-02-18 12:00:00,279.9,26.0,0.5
3,heathrow,32.55,35.85,10.0,2012-02-18 06:00:00,273.9,29.0,7.0


## Several files

A list of geopoints files (or a glob pattern or a directory) gives a single data object. Its `to_pandas()` stacks the data of all the files, and its featurelists contain the features of all the files, in order.

In [14]:
ds = ekd.from_source("file", ["geopointset.gpts", "ncols_stnid.gpt"])
ds

path,"['geopointset.gpts', 'ncols_stnid.gpt']"
types,"pandas, xarray, featurelist"


In [15]:
ds.to_pandas()

,latitude,longitude,level,date,value,stnid,_v1,_v2,_v3
0,69.6523,18.9057,0.0,2013-05-12 00:00:00,100869.86250,NaN,NaN,NaN,NaN
1,63.4882,10.8795,0.0,2013-05-12 00:00:00,100282.33920,NaN,NaN,NaN,NaN
2,63.5657,10.6940,0.0,2013-05-12 00:00:00,100241.16660,NaN,NaN,NaN,NaN
3,61.2928,5.0443,0.0,2013-05-12 00:00:00,99852.18932,NaN,NaN,NaN,NaN
4,60.8200,23.5000,0.0,2013-05-12 06:00:00,101045.80000,NaN,NaN,NaN,NaN
5,55.0100,8.4100,0.0,2013-05-13 00:00:00,100949.18090,NaN,NaN,NaN,NaN
6,54.3300,8.6200,0.0,2013-05-13 00:00:00,101027.91010,NaN,NaN,NaN,NaN
7,53.7100,7.1500,0.0,2013-05-13 00:00:00,100846.61900,NaN,NaN,NaN,NaN
8,32.5500,35.8500,10.0,2012-02-18 06:00:00,NaN,heathrow,273.9,29.0,7.0
9,31.7200,35.9800,20.0,2012-02-18 18:00:00,NaN,oxford,274.9,28.0,1.0


In [16]:
len(ds.to_featurelist())

4

## Note on the `.csv` extension